# 08 — Team Report
**Purpose:** Scoped analysis and recommendations for a single team.
**Inputs:** All intermediate outputs from notebooks 02–07.
**Outputs:** Self-contained team-specific summary.

In [ ]:
# === CONFIGURATION ===
# Set the team identifier. This notebook can be run via papermill with:
#   papermill 08_team_report.ipynb output.ipynb -p TEAM_ID "team_name"
TEAM_ID = "ALL"  # Change this, or override via papermill

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import seaborn as sns

from buildanalysis.build import compute_critical_path
from buildanalysis.git import compute_file_to_target_map, infer_team_assignments
from buildanalysis.graphs import build_dependency_graph
from buildanalysis.loading import BuildDataset
from buildanalysis.recommend import (
    build_pareto_frontier,
    format_recommendation_summary,
    score_dependency_interventions,
    score_header_interventions,
    score_target_split_interventions,
)
from buildanalysis.types import AnalysisScope

warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (10, 6), "figure.dpi": 100})

DATA_DIR = Path("../../data/processed")
INTERMEDIATE_DIR = DATA_DIR / "intermediate"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

ds = BuildDataset(DATA_DIR, intermediate_dir=INTERMEDIATE_DIR, validate=False)
tm = ds.target_metrics
el = ds.edge_list
fm = ds.file_metrics

bg = build_dependency_graph(tm, el, direct_only=True)
cp_result = compute_critical_path(bg, tm)

print(f"Targets: {len(tm):,}  |  Edges: {len(el):,}  |  Files: {len(fm):,}")
print(f"Critical path: {cp_result.total_time_s:.1f}s  ({len(cp_result.path)} targets)")

## 1. Resolve Team Scope

In [ ]:
# Load team assignments
HAS_GIT = ds.has_file("git_commit_log")

try:
    team_df = ds.load_intermediate("target_ownership")
    team_col = "primary_team" if "primary_team" in team_df.columns else "top_contributor"
    print(f"Loaded target_ownership (using column '{team_col}')")
except FileNotFoundError:
    if HAS_GIT:
        file_to_target = compute_file_to_target_map(fm)
        team_df = infer_team_assignments(ds.git_commit_log, file_to_target)
        team_col = "primary_team"
        print("Inferred team assignments from git log")
    else:
        team_df = pd.DataFrame({"cmake_target": tm["cmake_target"], "primary_team": "unknown"})
        team_col = "primary_team"
        print("No git data or target_ownership — all targets assigned to 'unknown'")

team_map = team_df.set_index("cmake_target")[team_col].to_dict()
all_teams = sorted(team_df[team_col].unique())
print(f"Teams found: {len(all_teams)}")

In [ ]:
# If TEAM_ID == "ALL", list available teams and stop
if TEAM_ID == "ALL":
    print("AVAILABLE TEAMS")
    print("=" * 60)
    team_counts = team_df.groupby(team_col).size().sort_values(ascending=False)
    for team_name, count in team_counts.items():
        print(f"  {team_name:<40s}  {count:>4} targets")
    print()
    print("Re-run this notebook with a specific TEAM_ID, e.g.:")
    print(f'  papermill 08_team_report.ipynb reports/team_NAME.ipynb -p TEAM_ID "NAME"')
    raise SystemExit("Set TEAM_ID to a specific team to generate a report.")

In [ ]:
# Build AnalysisScope for the team
team_targets_series = team_df[team_df[team_col] == TEAM_ID]["cmake_target"]
if team_targets_series.empty:
    raise ValueError(f"Team '{TEAM_ID}' not found. Available: {all_teams}")

team_targets = frozenset(team_targets_series)
scope = AnalysisScope(
    targets=team_targets,
    teams=frozenset([TEAM_ID]),
    label=TEAM_ID,
)

# Target type breakdown
team_tm = scope.filter_targets(tm)
type_counts = team_tm["target_type"].value_counts()

print(f"TEAM: {TEAM_ID}")
print(f"Owned targets: {len(team_targets)}")
print(f"Target types:")
for tt, count in type_counts.items():
    print(f"  {tt}: {count}")

## 2. Team’s Targets Overview

In [ ]:
# Load optional intermediates for layer/community info
try:
    graph_analysis = ds.load_intermediate("graph_analysis")
except FileNotFoundError:
    graph_analysis = None

try:
    communities_df = ds.load_intermediate("communities")
except FileNotFoundError:
    communities_df = None

# Build overview table
overview_cols = ["cmake_target", "target_type"]
for col in ["file_count", "code_lines_total", "total_build_time_ms", "codegen_ratio"]:
    if col in team_tm.columns:
        overview_cols.append(col)

overview = team_tm[overview_cols].copy()

# Add community if available
if communities_df is not None:
    comm_map = communities_df.set_index("cmake_target")["community"].to_dict()
    overview["community"] = overview["cmake_target"].map(comm_map)

# Add layer if available
if graph_analysis is not None and "layer" in graph_analysis.columns:
    layer_map = graph_analysis.set_index("cmake_target")["layer"].to_dict()
    overview["layer"] = overview["cmake_target"].map(layer_map)

overview = overview.sort_values("total_build_time_ms", ascending=False) if "total_build_time_ms" in overview.columns else overview.sort_values("cmake_target")
overview.index = range(1, len(overview) + 1)

print(f"TEAM '{TEAM_ID}' — TARGET OVERVIEW")
print("=" * 120)
print(overview.to_string())

# Build time comparison
if "total_build_time_ms" in team_tm.columns:
    team_build_ms = team_tm["total_build_time_ms"].sum()
    total_build_ms = tm["total_build_time_ms"].sum()
    pct = team_build_ms / total_build_ms * 100 if total_build_ms > 0 else 0
    print(f"\nTeam build time: {team_build_ms / 1000:,.1f}s ({team_build_ms / 60_000:,.1f} min)")
    print(f"Total codebase:  {total_build_ms / 1000:,.1f}s ({total_build_ms / 60_000:,.1f} min)")
    print(f"Team fraction:   {pct:.1f}%")

## 3. Team’s Build Cost Breakdown

In [ ]:
# Phase breakdown across team targets
phase_cols = {
    "compile_time_sum_ms": "Compile",
    "codegen_time_ms": "Codegen",
    "link_time_ms": "Link",
    "archive_time_ms": "Archive",
}
available_phases = {col: label for col, label in phase_cols.items() if col in team_tm.columns}

if available_phases:
    phase_totals = {label: team_tm[col].sum() for col, label in available_phases.items()}
    phase_series = pd.Series(phase_totals).sort_values(ascending=False)
    phase_series = phase_series[phase_series > 0]

    if not phase_series.empty:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Pie chart
        axes[0].pie(phase_series, labels=phase_series.index, autopct="%1.0f%%", startangle=90)
        axes[0].set_title(f"Build Cost Breakdown — {TEAM_ID}")

        # Bar chart
        axes[1].barh(phase_series.index, phase_series.values / 1000, color="steelblue", edgecolor="white", linewidth=0.3)
        axes[1].set_xlabel("Time (s)")
        axes[1].set_title("Build Phase Times")

        fig.tight_layout()
        plt.show()

        print("PHASE BREAKDOWN")
        print("=" * 40)
        for label, val in phase_series.items():
            print(f"  {label:<12s}  {val / 1000:>8.1f}s")
else:
    print("Phase columns not available in target_metrics.")

In [ ]:
# GCC phase breakdown for team's files
team_fm = scope.filter_targets(fm) if "cmake_target" in fm.columns else fm[fm["source_file"].isin(scope.files or [])]

gcc_phase_cols = {
    "gcc_parse_time_ms": "Parsing",
    "gcc_template_time_ms": "Template instantiation",
    "gcc_codegen_time_ms": "Code generation",
    "gcc_opt_time_ms": "Optimisation",
}
available_gcc = {col: label for col, label in gcc_phase_cols.items() if col in team_fm.columns}

if available_gcc:
    gcc_totals = {label: team_fm[col].sum() for col, label in available_gcc.items()}
    gcc_series = pd.Series(gcc_totals).sort_values(ascending=False)
    gcc_total = gcc_series.sum()

    print("GCC PHASE BREAKDOWN (team files)")
    print("=" * 50)
    for label, val in gcc_series.items():
        pct = val / gcc_total * 100 if gcc_total > 0 else 0
        print(f"  {label:<25s}  {val / 1000:>8.1f}s  ({pct:.0f}%)")
else:
    print("GCC phase columns not available in file_metrics.")

In [ ]:
# Top 5 most expensive targets with file drill-down
if "total_build_time_ms" in team_tm.columns:
    top5 = team_tm.nlargest(5, "total_build_time_ms")

    print("TOP 5 MOST EXPENSIVE TARGETS")
    print("=" * 100)
    for _, trow in top5.iterrows():
        target = trow["cmake_target"]
        bt = trow["total_build_time_ms"]
        print(f"\n  {target}  ({bt / 1000:.1f}s)")

        # Top 3 files within this target
        if "cmake_target" in fm.columns and "compile_time_ms" in fm.columns:
            target_files = fm[fm["cmake_target"] == target].nlargest(3, "compile_time_ms")
            for _, frow in target_files.iterrows():
                print(f"    {frow['source_file']:<70s}  {frow['compile_time_ms']:>8.0f}ms")

## 4. Team’s Dependency Footprint

In [ ]:
# Direct dependencies and dependants
direct_el = el[el["is_direct"]].copy()

# Outgoing: team targets that depend on external targets
team_deps_out = direct_el[
    direct_el["source_target"].isin(team_targets) & ~direct_el["dest_target"].isin(team_targets)
]
# Incoming: external targets that depend on team targets
team_deps_in = direct_el[
    ~direct_el["source_target"].isin(team_targets) & direct_el["dest_target"].isin(team_targets)
]

print(f"Direct dependencies on other teams' targets:  {len(team_deps_out)}")
print(f"Direct dependants from other teams' targets:  {len(team_deps_in)}")

# Transitive dependency set
transitive_deps = set()
for t in team_targets:
    if t in bg.graph:
        transitive_deps |= nx.descendants(bg.graph, t)
# Exclude team's own targets
external_transitive = transitive_deps - team_targets
total_build_set = team_targets | transitive_deps

print(f"\nTransitive dependency set: {len(total_build_set)} targets ")
print(f"  ({len(total_build_set) / bg.n_targets * 100:.1f}% of codebase)")
print(f"  Own: {len(team_targets)}, External deps: {len(external_transitive)}")

In [ ]:
# External dependencies grouped by owning team
team_deps_out_with_team = team_deps_out.copy()
team_deps_out_with_team["dep_team"] = team_deps_out_with_team["dest_target"].map(team_map).fillna("unknown")

deps_by_team = (
    team_deps_out_with_team.groupby("dep_team")
    .agg(n_edges=("dest_target", "size"), targets=("dest_target", "nunique"))
    .sort_values("n_edges", ascending=False)
)

print(f"DEPENDENCIES ON OTHER TEAMS ({TEAM_ID} depends on)")
print("=" * 60)
print(deps_by_team.to_string())

# External dependants grouped by owning team
team_deps_in_with_team = team_deps_in.copy()
team_deps_in_with_team["dep_team"] = team_deps_in_with_team["source_target"].map(team_map).fillna("unknown")

dependants_by_team = (
    team_deps_in_with_team.groupby("dep_team")
    .agg(n_edges=("source_target", "size"), targets=("source_target", "nunique"))
    .sort_values("n_edges", ascending=False)
)

print(f"\nDEPENDANTS FROM OTHER TEAMS (teams that depend on {TEAM_ID})")
print("=" * 60)
print(dependants_by_team.to_string())

In [ ]:
# PUBLIC dependencies that could potentially be PRIVATE
if "cmake_visibility" in direct_el.columns:
    team_public = direct_el[
        direct_el["source_target"].isin(team_targets)
        & (direct_el["cmake_visibility"] == "PUBLIC")
    ]
    team_private = direct_el[
        direct_el["source_target"].isin(team_targets)
        & (direct_el["cmake_visibility"] == "PRIVATE")
    ]

    print(f"Team's PUBLIC dependencies:  {len(team_public)}")
    print(f"Team's PRIVATE dependencies: {len(team_private)}")
    if len(team_public) > 0:
        print(f"\nPUBLIC deps are candidates for narrowing to PRIVATE if dependants")
        print(f"do not actually use the transitive headers.")
        print(f"\nPUBLIC dependencies:")
        for _, row in team_public.iterrows():
            print(f"  {row['source_target']} -> {row['dest_target']}")
else:
    print("cmake_visibility not available in edge list.")

## 5. Team’s Header Impact

In [ ]:
try:
    header_impact = ds.load_intermediate("header_impact")
except FileNotFoundError:
    header_impact = None

if header_impact is not None:
    # Determine which headers belong to team targets via file_metrics
    if "cmake_target" in fm.columns:
        team_files = set(fm[fm["cmake_target"].isin(team_targets)]["source_file"])
    else:
        team_files = set()

    # Also check header_metrics for target ownership
    try:
        hm = ds.header_metrics
        team_headers_via_hm = set(hm[hm["cmake_target"].isin(team_targets)]["header_file"])
    except (FileNotFoundError, AttributeError):
        team_headers_via_hm = set()

    team_header_files = team_files | team_headers_via_hm

    # Headers owned by this team
    owned_headers = header_impact[header_impact["file"].isin(team_header_files)]
    owned_headers = owned_headers.sort_values("impact_score", ascending=False)

    print(f"TOP 10 HIGHEST-IMPACT HEADERS OWNED BY {TEAM_ID}")
    print("(candidates for refactoring to reduce rebuild cost)")
    print("=" * 100)
    display_cols = [c for c in ["file", "transitive_fan_in", "source_size_bytes", "n_commits", "impact_score"] if c in owned_headers.columns]
    if not owned_headers.empty:
        print(owned_headers[display_cols].head(10).to_string(index=False))
    else:
        print("  No headers found for this team in header_impact data.")

    # External headers that team's source files include heavily
    # These are headers NOT owned by this team but present in header_impact
    external_headers = header_impact[~header_impact["file"].isin(team_header_files)]

    # Cross-reference with team's include trees if header_edges available
    try:
        he = ds.header_edges
        # Find headers included by team's source files
        team_source_files = set(fm[fm["cmake_target"].isin(team_targets)]["source_file"]) if "cmake_target" in fm.columns else set()
        team_includes = he[he["source_file"].isin(team_source_files)]["included"].value_counts()
        external_used = external_headers[external_headers["file"].isin(team_includes.index)].copy()
        external_used["team_include_count"] = external_used["file"].map(team_includes).fillna(0).astype(int)
        external_used = external_used.sort_values("team_include_count", ascending=False)
    except (FileNotFoundError, AttributeError):
        # Fall back to just showing top external headers by impact
        external_used = external_headers.copy()

    print(f"\nTOP 10 EXTERNAL HEADERS DEPENDED ON BY {TEAM_ID}")
    print("(candidates for forward declaration or interface changes)")
    print("=" * 100)
    ext_display = [c for c in ["file", "transitive_fan_in", "source_size_bytes", "impact_score", "team_include_count"] if c in external_used.columns]
    if not external_used.empty:
        print(external_used[ext_display].head(10).to_string(index=False))
    else:
        print("  No external header dependencies found.")
else:
    print("header_impact.parquet not available — skipping header analysis.")

## 6. Team’s Modularity Position

In [ ]:
if communities_df is not None:
    comm_map = communities_df.set_index("cmake_target")["community"].to_dict()
    team_communities = [
        comm_map[t] for t in team_targets if t in comm_map
    ]

    if team_communities:
        comm_counts = pd.Series(team_communities).value_counts()
        dominant_comm = comm_counts.index[0]
        dominant_frac = comm_counts.iloc[0] / len(team_communities)

        print(f"MODULARITY POSITION — {TEAM_ID}")
        print("=" * 60)
        print(f"Communities spanned: {len(comm_counts)}")
        for comm_id, count in comm_counts.items():
            pct = count / len(team_communities) * 100
            print(f"  Community {comm_id}: {count} targets ({pct:.0f}%)")

        if len(comm_counts) == 1:
            print(f"\nTeam is concentrated in a single community — good for subset builds.")
        elif dominant_frac > 0.8:
            print(f"\nTeam is mostly in community {dominant_comm} ({dominant_frac:.0%}) with minor scatter.")
        else:
            print(f"\nTeam is scattered across {len(comm_counts)} communities — subset builds will pull in more code.")
    else:
        print("No team targets found in community assignments.")
else:
    print("communities.parquet not available — skipping modularity analysis.")

In [ ]:
# Feature configurations: what features include this team's targets?
try:
    feature_configs = ds.load_intermediate("feature_configurations")
except FileNotFoundError:
    feature_configs = None

if feature_configs is not None:
    team_features = []
    for _, row in feature_configs.iterrows():
        own_targets = set(json.loads(row["own_target_list"]))
        overlap = own_targets & team_targets
        if overlap:
            team_features.append({
                "feature_id": row["feature_id"],
                "own_targets": row["own_targets"],
                "total_build_set": row["total_build_set"],
                "build_fraction": row["build_fraction"],
                "team_targets_in_feature": len(overlap),
            })

    if team_features:
        tf_df = pd.DataFrame(team_features)
        print(f"\nFEATURES CONTAINING {TEAM_ID}'S TARGETS")
        print("=" * 90)
        print(tf_df.to_string(index=False, float_format="%.3f"))

        # Minimum build fraction: smallest feature that includes team targets
        min_feature = tf_df.loc[tf_df["build_fraction"].idxmin()]
        print(f"\nSmallest feature including team targets: feature {min_feature['feature_id']}")
        print(f"  Build fraction: {min_feature['build_fraction']:.1%} of codebase")
    else:
        print(f"No features contain targets owned by {TEAM_ID}.")
else:
    print("feature_configurations.parquet not available — skipping feature analysis.")

## 7. Team-Specific Recommendations

In [ ]:
# Try loading pre-computed recommendations from notebook 07
try:
    all_recs = ds.load_intermediate("recommendations")
    # Filter to interventions affecting this team's targets
    def affects_team(targets_affected):
        if isinstance(targets_affected, str):
            try:
                targets_affected = json.loads(targets_affected)
            except (json.JSONDecodeError, TypeError):
                targets_affected = [targets_affected]
        if not isinstance(targets_affected, list):
            return False
        return bool(set(targets_affected) & team_targets)

    team_recs = all_recs[all_recs["targets_affected"].apply(affects_team)].copy()
    team_recs = team_recs.sort_values("impact_ms", ascending=False)

    print(f"RECOMMENDATIONS FOR {TEAM_ID}")
    print("=" * 100)
    if not team_recs.empty:
        team_recs.index = range(1, len(team_recs) + 1)
        display_cols = ["type", "description", "impact_ms", "effort_days", "confidence", "rationale"]
        display_cols = [c for c in display_cols if c in team_recs.columns]
        print(team_recs[display_cols].to_string())
    else:
        print(f"No specific recommendations from notebook 07 affect {TEAM_ID}'s targets.")
        print(f"\nGeneral advice based on team profile:")
        if len(team_deps_out) < 5:
            print(f"  - Your targets are well-contained with low coupling — focus on header hygiene.")
        if "total_build_time_ms" in team_tm.columns:
            team_pct = team_tm["total_build_time_ms"].sum() / tm["total_build_time_ms"].sum() * 100
            if team_pct < 5:
                print(f"  - Your team accounts for only {team_pct:.1f}% of build time — low priority for optimisation.")
            else:
                print(f"  - Your team accounts for {team_pct:.1f}% of build time — consider reviewing the most expensive targets above.")

    HAS_PRECOMPUTED = True

except FileNotFoundError:
    HAS_PRECOMPUTED = False
    print("recommendations.parquet not found — generating team-scoped recommendations from scratch.")

In [ ]:
# Generate recommendations from scratch if pre-computed ones are unavailable
if not HAS_PRECOMPUTED:
    from buildanalysis.recommend import Intervention

    all_interventions: list[Intervention] = []

    # Header interventions (scoped to team headers)
    if header_impact is not None and team_header_files:
        team_hi = header_impact[header_impact["file"].isin(team_header_files)].copy()
        if not team_hi.empty:
            include_amp = pd.DataFrame(columns=["file", "direct_includes", "transitive_includes", "amplification_ratio"])
            hi_interventions = score_header_interventions(team_hi, include_amp, top_n=10)
            all_interventions.extend(hi_interventions)

    # Target split interventions (scoped to team targets on CP)
    split_interventions = score_target_split_interventions(team_tm, cp_result, top_n=5)
    all_interventions.extend(split_interventions)

    # Dependency interventions (edges from team targets)
    team_el = el[el["source_target"].isin(team_targets)].copy()
    if not team_el.empty:
        el_renamed = team_el.rename(columns={"source_target": "source", "dest_target": "dependency"})
        dep_interventions = score_dependency_interventions(bg, tm, cp_result, el_renamed, top_n=10)
        all_interventions.extend(dep_interventions)

    if all_interventions:
        team_pareto = build_pareto_frontier(all_interventions)
        team_recs = team_pareto.sort_values("impact_ms", ascending=False)
        team_recs.index = range(1, len(team_recs) + 1)

        print(f"GENERATED RECOMMENDATIONS FOR {TEAM_ID}")
        print("=" * 100)
        display_cols = [c for c in ["type", "description", "impact_ms", "effort_days", "confidence", "pareto_optimal"] if c in team_recs.columns]
        print(team_recs[display_cols].to_string())
    else:
        team_recs = pd.DataFrame()
        print(f"No interventions generated for {TEAM_ID}.")
        print(f"General advice: focus on header hygiene and monitoring build time trends.")

## 8. Team Summary Card

In [ ]:
# Build summary card
n_executables = (team_tm["target_type"] == "executable").sum() if "target_type" in team_tm.columns else 0
n_libraries = len(team_tm) - n_executables

team_build_s = team_tm["total_build_time_ms"].sum() / 1000 if "total_build_time_ms" in team_tm.columns else 0
team_build_min = team_build_s / 60
total_build_s = tm["total_build_time_ms"].sum() / 1000 if "total_build_time_ms" in tm.columns else 1
team_build_pct = team_build_s / total_build_s * 100 if total_build_s > 0 else 0

dep_footprint = len(total_build_set)
dep_footprint_pct = dep_footprint / bg.n_targets * 100 if bg.n_targets > 0 else 0

# Community info
if communities_df is not None and team_communities:
    comm_counts_s = pd.Series(team_communities).value_counts()
    dominant = comm_counts_s.index[0]
    community_str = f"primarily community {dominant}"
else:
    community_str = "not available"

# Feature build fraction
if feature_configs is not None and team_features:
    min_bf = min(f["build_fraction"] for f in team_features)
    feature_str = f"{min_bf:.0%} of codebase"
else:
    feature_str = "not available"

# Top recommendations
rec_lines = []
if isinstance(team_recs, pd.DataFrame) and not team_recs.empty:
    top_recs = team_recs.head(3)
    for i, (_, row) in enumerate(top_recs.iterrows(), 1):
        impact_s = row["impact_ms"] / 1000
        effort = row["effort_days"]
        rec_lines.append(f"{i}. {row['description']} (est. -{impact_s:.0f}s, ~{effort:.1f} days)")

    total_impact_s = top_recs["impact_ms"].sum() / 1000
else:
    total_impact_s = 0

# Print summary card
print(f"TEAM REPORT: {TEAM_ID}")
print("=" * 60)
print(f"Targets: {len(team_targets)} ({n_executables} executables, {n_libraries} libraries)")
print(f"Build time: {team_build_min:.1f} min ({team_build_pct:.1f}% of total)")
print(f"Dependency footprint: {dep_footprint} targets ({dep_footprint_pct:.1f}% of codebase)")
print(f"Community: {community_str}")
print(f"Feature build fraction: {feature_str}")

if rec_lines:
    print(f"\nTOP {len(rec_lines)} RECOMMENDATIONS:")
    for line in rec_lines:
        print(f"  {line}")
    print(f"\nTOTAL ESTIMATED IMPACT: -{total_impact_s:.0f}s on build time")
else:
    print(f"\nNo specific recommendations for this team.")